In [ ]:
# ============================================================
# ydata-profiling for DataCo Smart Supply Chain — All 3 Files
# ============================================================
# Install: pip install ydata-profiling pandas
# Run:     python dataco_profiling.py
# Outputs: 3 HTML reports in the same folder
# ============================================================
 
import pandas as pd
# from ydata_profiling import ProfileReport
import os
import warnings
warnings.filterwarnings("ignore")
 


In [ ]:
# ─────────────────────────────────────────
# CONFIG — update paths if needed
# ─────────────────────────────────────────
DATA_DIR = "./"   # folder where your 3 files live
 
FILE_MAIN   = os.path.join(DATA_DIR, "DataCoSupplyChainDataset.csv")
FILE_LOGS   = os.path.join(DATA_DIR, "tokenized_access_logs.csv")
FILE_DESC   = os.path.join(DATA_DIR, "DescriptionDataCoSupplyChain.csv")
 
OUT_MAIN    = "report_main_dataset.html"
OUT_LOGS    = "report_access_logs.html"
OUT_DESC    = "report_description.html"

In [ ]:
# HELPER: safe loader with encoding fallback
# ─────────────────────────────────────────
def load_csv(path, name, nrows=None):
    print(f"\n{'='*55}")
    print(f"  Loading: {name}")
    print(f"{'='*55}")
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, encoding=enc, nrows=nrows,
                             on_bad_lines="skip", low_memory=False)
            print(f"  Encoding used : {enc}")
            print(f"  Shape         : {df.shape[0]:,} rows x {df.shape[1]} columns")
            print(f"  Memory        : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
            return df
        except Exception as e:
            print(f"  Tried {enc}: {e}")
    raise RuntimeError(f"Could not load {path}")
 

In [ ]:
# ─────────────────────────────────────────
# FILE 1 — Main structured dataset
# ─────────────────────────────────────────
df_main = load_csv(FILE_MAIN, "DataCoSupplyChainDataset.csv")
 
# Parse date columns so profiling detects them as datetime
for col in ["order date (DateOrders)", "shipping date (DateOrders)"]:
    if col in df_main.columns:
        df_main[col] = pd.to_datetime(df_main[col], errors="coerce")
 
# Mark the binary target so profiling highlights it
if "Late_delivery_risk" in df_main.columns:
    df_main["Late_delivery_risk"] = df_main["Late_delivery_risk"].astype("category")
 
print("\n  Generating profile report for File 1 (this may take ~2 min)...")
profile_main = ProfileReport(
    df_main,
    title="DataCo Supply Chain — Main Dataset",
    dataset={
        "description": "180k+ order records with delivery, shipping, and financial features.",
        "url": "https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis"
    },
    # Correlation: include Spearman + Cramér's V for mixed feature types
    correlations={
        "pearson":  {"calculate": True},
        "spearman": {"calculate": True},
        "cramers":  {"calculate": True},
        "phi_k":    {"calculate": False},
    },
    # Missing values diagram
    missing_diagrams={
        "bar":    True,
        "matrix": True,
        "heatmap": True,
    },
    # Sample table at the top of the report
    samples={"head": 10, "tail": 5},
    # Interactions: plot top feature pairs (slow on 50+ cols — set to False to skip)
    interactions={"continuous": False},
    explorative=True,
    progress_bar=True,
)
profile_main.to_file(OUT_MAIN)
print(f"  Saved: {OUT_MAIN}")

In [ ]:
# ─────────────────────────────────────────
# FILE 2 — Tokenized access logs
# Sampling: file is 90 MB — profile 50k rows
# for speed; increase nrows for full scan
# ─────────────────────────────────────────
LOGS_SAMPLE = 50_000   # set to None to profile everything (slow)
 
df_logs = load_csv(FILE_LOGS, "tokenized_access_logs.csv", nrows=LOGS_SAMPLE)
 
print(f"\n  Profiling {min(LOGS_SAMPLE, len(df_logs)):,} rows of access logs...")
profile_logs = ProfileReport(
    df_logs,
    title=f"DataCo — Tokenized Access Logs (sample: {len(df_logs):,} rows)",
    dataset={
        "description": (
            "Clickstream / web access log — tokenized sequences of user/system "
            "actions. Unstructured secondary signal for anomaly detection."
        )
    },
    # For text-heavy token columns use minimal correlations
    correlations={
        "pearson":  {"calculate": True},
        "spearman": {"calculate": False},
        "cramers":  {"calculate": False},
        "phi_k":    {"calculate": False},
    },
    missing_diagrams={"bar": True, "matrix": False, "heatmap": False},
    samples={"head": 10, "tail": 5},
    interactions={"continuous": False},
    explorative=True,
    progress_bar=True,
)
profile_logs.to_file(OUT_LOGS)
print(f"  Saved: {OUT_LOGS}")